In [1]:
import spotipy
import os
from spotipy.oauth2 import SpotifyClientCredentials
from spotipy.oauth2 import SpotifyOAuth
from pprint import pprint
import pandas as pd
from dotenv import load_dotenv

In [2]:
'''
dotenv file should contain:
SPOTIPY_CLIENT_ID = "your client id"
SPOTIPY_CLIENT_SECRET = "your client secret"  
SPOTIPY_REDIRECT_URI = "http://localhost:8888/callback"
'''
load_dotenv("C:/apis/.env") # path to your dotenv file
client_id = os.getenv("SPOTIPY_CLIENT_ID")
client_secret = os.getenv("SPOTIPY_CLIENT_SECRET")
redirect_uri = os.getenv("SPOTIPY_REDIRECT_URI")

# This masks your secret keys before printing them, in case you are sharing this notebook:
def mask_secret(unmasked_chars, secret):
    masked_token = secret[:unmasked_chars] + '*' * (len(secret) - unmasked_chars*2) + secret[-unmasked_chars:]
    return masked_token

print(f"SPOTIPY_CLIENT_ID: {mask_secret(4, client_id)}")
print(f"SPOTIPY_CLIENT_SECRET: {mask_secret(4, client_secret)}")
print(f"SPOTIPY_REDIRECT_URI: {redirect_uri}")

SPOTIPY_CLIENT_ID: 3068************************2505
SPOTIPY_CLIENT_SECRET: 9945************************2daa
SPOTIPY_REDIRECT_URI: http://localhost:8888/callback


Checking all my playlists:

In [4]:
#sp = spotipy.Spotify(client_credentials_manager=SpotifyClientCredentials())
# This will authenticate you on your web browser:
sp = spotipy.Spotify(auth_manager=SpotifyOAuth(client_id=os.getenv("SPOTIPY_CLIENT_ID"),
                                               client_secret=os.getenv("SPOTIPY_CLIENT_SECRET"),
                                               redirect_uri=os.getenv("SPOTIPY_REDIRECT_URI"),
                                               scope="playlist-modify-public playlist-modify-private"))

user_id = "duhbeed"
if not user_id:
    raise ValueError("Spotify username is required.")

try:
    results = sp.user_playlists(user_id)
except spotipy.exceptions.SpotifyException as exc:
    print(f"Failed to fetch playlists: {exc}")
else:
    if not results["items"]:
        print("No public playlists found.")
    else:
        while True:
            for playlist in results["items"]:
                print(f"{playlist['name']} ({playlist['tracks']['total']} tracks)")
            if results["next"]:
                results = sp.next(results)
            else:
                break

⌛ Mad Cool 2025 - Jueves (orden horario) ⌛ (100 tracks)
My 2024 Playlist in a Bottle (8 tracks)
Best of 2024 (2/3) | Electronic & Hip-Hop (18 tracks)
Best of 2024 (1/3) | (Mostly) Rock (17 tracks)
Lo Mejor de 2024 (3/3) | de España y/o en castellano (15 tracks)
Dogs of TikTok, YouTube and Instagram (reddgr.com) - sorted by track popularity (43 tracks)
Low Festival 2024  (Sábado) 🌊 ¡Orden horario! ⏰ (100 tracks)
Low Festival 2024 🌊 (Domingo) ¡Orden horario! ⏰ (Domingo) (100 tracks)
Low Festival 2024 🌊 ¡Orden horario! ⏰ (Viernes) (100 tracks)
Talking to Chatbots (Reddgr) (International Playlist) (20 tracks)
Colección de podcasts de Reddgr (17 tracks)
Tomavistas 2024 (orden horario) (146 tracks)
Bands and Artists I've Seen Live (Sorted by artist popularity)  (478 tracks)
Reddgr Curated Podcasts (35 tracks)
Dogs of TikTok, YouTube and Instagram (reddgr.com) (44 tracks)
Talking to Chatbots (TTCB) (15 tracks)
DCODE 2022 by David (112 tracks)
Mad Cool 2022 (Jueves) (121 tracks)
2022 (55 track

Full dataframe with playlist metadata:

In [54]:
playlist_items = []
page = sp.user_playlists(user_id, limit=50)
while page:
    playlist_items.extend(page.get("items", []))
    page = sp.next(page) if page.get("next") else None

column_map = {
    "name": "name",
    "id": "playlist_id",
    # "owner.display_name": "owner_display_name",
    # "owner.id": "owner_id",
    # "public": "public",
    # "collaborative": "collaborative",
    "tracks.total": "tracks_total",
    "description": "description",
    "snapshot_id": "snapshot_id",
    "external_urls.spotify": "external_url",
    "uri": "uri",
    # "primary_color": "primary_color",
}

if playlist_items:
    playlists_raw = pd.json_normalize(playlist_items)
    available_cols = [col for col in column_map if col in playlists_raw.columns]
    playlists_df = (
        playlists_raw[available_cols]
        .rename(columns={col: column_map[col] for col in available_cols})
    )
    playlists_df = playlists_df[[column_map[col] for col in available_cols]]
else:
    playlists_df = pd.DataFrame(columns=list(column_map.values()))

display(playlists_df)

,name,playlist_id,tracks_total,description,snapshot_id,external_url,uri
0,⌛ Mad Cool 2025 - Jueves (orden horario) ⌛,0acg8XNM0LxaTeSan2U50T,100,100 pistas para preparar el primer día de Mad ...,AAAArqHP41rl5rCp+ib7PEnauUlV+gpj,https://open.spotify.com/playlist/0acg8XNM0Lxa...,spotify:playlist:0acg8XNM0LxaTeSan2U50T
1,My 2024 Playlist in a Bottle,1x5Tv4ITB7rq0OPcUB0NEo,8,A musical time capsule from the past has been ...,AAAAA7aIkhTuCJ9n9Lq8DK1+XwUBCz3o,https://open.spotify.com/playlist/1x5Tv4ITB7rq...,spotify:playlist:1x5Tv4ITB7rq0OPcUB0NEo
2,Best of 2024 (2/3) | Electronic & Hip-Hop,2Bqun7K2S7cAksiUFCQTEm,18,"Every year&#x27;s curated playlists, to be lis...",AAAARHBDRPQyajXV3u4Km8mNKaKRCOoS,https://open.spotify.com/playlist/2Bqun7K2S7cA...,spotify:playlist:2Bqun7K2S7cAksiUFCQTEm
3,Best of 2024 (1/3) | (Mostly) Rock,4AA6jg6T0PbzkVJZKyI5X9,17,"Every year&#x27;s curated playlists, to be lis...",AAAAU5ugERkxM+m+7k0rQJpTmzXlWlaz,https://open.spotify.com/playlist/4AA6jg6T0Pbz...,spotify:playlist:4AA6jg6T0PbzkVJZKyI5X9
4,Lo Mejor de 2024 (3/3) | de España y/o en cast...,1LyCSbuOeqglulGBKP8vSZ,15,Mi sesión de canciones en castellano y de arti...,AAAAVf8WUm9jTLDdOsdAdAumSdFK86qP,https://open.spotify.com/playlist/1LyCSbuOeqgl...,spotify:playlist:1LyCSbuOeqglulGBKP8vSZ
...,...,...,...,...,...,...,...
148,Totally Random Playlist,0AMHQuovdwMQ36DQip0UpQ,20,,AAAAV/jDxNUkR+/zNbkrCdNguzd06UDz,https://open.spotify.com/playlist/0AMHQuovdwMQ...,spotify:playlist:0AMHQuovdwMQ36DQip0UpQ
149,Video Games,0RGDLwNwo4p80AZgms2ZsE,10,,AAAAD9ZjUNOEJc/sSg+Naa7Cv6/oAZoJ,https://open.spotify.com/playlist/0RGDLwNwo4p8...,spotify:playlist:0RGDLwNwo4p80AZgms2ZsE
150,Violence,6AUxha1PnIg6mvvklt0j6n,11,,AAAAD75grCTf+eBiHcDpAZ/bJTKw9g26,https://open.spotify.com/playlist/6AUxha1PnIg6...,spotify:playlist:6AUxha1PnIg6mvvklt0j6n
151,Weezer 10,0erlbFfIKEV4eMJEtKhuoV,10,,AAAAHSKE24JOgaiS2tjbSwNI3FtEm/sr,https://open.spotify.com/playlist/0erlbFfIKEV4...,spotify:playlist:0erlbFfIKEV4eMJEtKhuoV


In [55]:
playlists_df.sample(5)

,name,playlist_id,tracks_total,description,snapshot_id,external_url,uri
41,Best of 2019 (5/6),6fRj9kFDAUejTtCPcah0RE,14,"My 100-song selection of every year, split int...",AAAAVBu+2mpHayn4WS+/u3S1gFuUtv6o,https://open.spotify.com/playlist/6fRj9kFDAUej...,spotify:playlist:6fRj9kFDAUejTtCPcah0RE
79,Best of 2015 (6/6) (Spanish/Spain),0T9gSIElSSL75ffgc5UZXD,18,,AAAAhmHmbGP5ePl63ATTPEkuCDpvZAgb,https://open.spotify.com/playlist/0T9gSIElSSL7...,spotify:playlist:0T9gSIElSSL75ffgc5UZXD
126,Italo-Disco,2qajcUJ7x242tTuBpEKUKx,6,,AAAACJ3AMtOATZSkibIYfNeji+dRKPOP,https://open.spotify.com/playlist/2qajcUJ7x242...,spotify:playlist:2qajcUJ7x242tTuBpEKUKx
64,Best of 2016 (6/6) (Spanish/Spain),1rEvf5o519fTAucn1BR3sM,18,,AAAAcJlTWx0P1bui4ckQCRl6Pb/a+sbK,https://open.spotify.com/playlist/1rEvf5o519fT...,spotify:playlist:1rEvf5o519fTAucn1BR3sM
106,Dcode 2013,1lnGc7yUY3sdgBkPlWL5Nu,21,,AAAARU6ARNt2+UjFiETMLyqDAt9g6Ime,https://open.spotify.com/playlist/1lnGc7yUY3sd...,spotify:playlist:1lnGc7yUY3sdgBkPlWL5Nu


In [60]:
filtered_playlists = (
    playlists_df[playlists_df["name"].str.lower().str.startswith(("best of", "lo mejor de"))]
    .reset_index(drop=True)
)

filtered_playlists = filtered_playlists.assign(
    year=pd.to_numeric(
        filtered_playlists["name"].str.extract(r"\b(2[0-9]{3})\b", expand=False),
        errors="coerce"
    )
)
main_cols = ["name", "year", "tracks_total", "description"]
cols = main_cols + [col for col in filtered_playlists.columns if col not in main_cols]
filtered_playlists = filtered_playlists[cols]

filtered_playlists = filtered_playlists.sort_values("year", ascending=True, na_position="last")
display(filtered_playlists)

,name,year,tracks_total,description,playlist_id,snapshot_id,external_url,uri
19,Best of 2010,2010,102,,5iBmxKUnNKzEZoYXxHYC2r,AAABUvdiqXREh7R4u2TxzLIgINVccq2z,https://open.spotify.com/playlist/5iBmxKUnNKzE...,spotify:playlist:5iBmxKUnNKzEZoYXxHYC2r
20,Best of 2011,2011,101,,2lMwz55DQSDpnNed34XFRa,AAACCO8tlBBDSkBWi+zh+FUIVowCy5pz,https://open.spotify.com/playlist/2lMwz55DQSDp...,spotify:playlist:2lMwz55DQSDpnNed34XFRa
71,Best of 2012 (bonus tracks),2012,23,,4jVzmc44I1IfrvgB2gaCIV,AAABztHil5w7vv0K1kpcb/CTmBbaf6u/,https://open.spotify.com/playlist/4jVzmc44I1If...,spotify:playlist:4jVzmc44I1IfrvgB2gaCIV
69,Best of 2012 (4/5) (electronic/hip-hop),2012,20,,73kk51cnXJvlsLp2LOi6wD,AAAAQvlfSNVRg/p7A6FQ20ae2iLiYYWA,https://open.spotify.com/playlist/73kk51cnXJvl...,spotify:playlist:73kk51cnXJvlsLp2LOi6wD
68,Best of 2012 (3/5) (experimental/other),2012,20,,4KYIK7rzuEO96twgvOynvM,AAAAVUe3z/AAcc+u3FJm1qbZaOKUUM1F,https://open.spotify.com/playlist/4KYIK7rzuEO9...,spotify:playlist:4KYIK7rzuEO96twgvOynvM
...,...,...,...,...,...,...,...,...
4,Best of 2023 (2/3),2023,18,"In this year&#x27;s collection, I&#x27;ve sele...",4XOMosM5oObvrcQVFCamFj,AAAAN2HQJgGJ1y7a9q0OzXqWJ9a+kB5H,https://open.spotify.com/playlist/4XOMosM5oObv...,spotify:playlist:4XOMosM5oObvrcQVFCamFj
3,Best of 2023 (1/3),2023,18,"In this year&#x27;s collection, I&#x27;ve sele...",34wg9M1ElBgos2l6QGExCd,AAAAPCLp2VtHkE0rm+UTkhq9PEIEf1zL,https://open.spotify.com/playlist/34wg9M1ElBgo...,spotify:playlist:34wg9M1ElBgos2l6QGExCd
2,Lo Mejor de 2024 (3/3) | de España y/o en cast...,2024,15,Mi sesión de canciones en castellano y de arti...,1LyCSbuOeqglulGBKP8vSZ,AAAAVf8WUm9jTLDdOsdAdAumSdFK86qP,https://open.spotify.com/playlist/1LyCSbuOeqgl...,spotify:playlist:1LyCSbuOeqglulGBKP8vSZ
1,Best of 2024 (1/3) | (Mostly) Rock,2024,17,"Every year&#x27;s curated playlists, to be lis...",4AA6jg6T0PbzkVJZKyI5X9,AAAAU5ugERkxM+m+7k0rQJpTmzXlWlaz,https://open.spotify.com/playlist/4AA6jg6T0Pbz...,spotify:playlist:4AA6jg6T0PbzkVJZKyI5X9
